# Second judge — cross-family Φ and judge–judge agreement

---
## 1 — Setup

In [1]:
!git pull

Already up to date.


In [ ]:
# No torch, no transformers. Seconds, not minutes.
!pip install -q openai==2.41.1 PyYAML==6.0.3 python-dotenv==1.2.2 numpy scipy

In [ ]:
import getpass, logging, os
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)   # one INFO line per call otherwise

In [ ]:
%cd /home/prnamhr/projects/Style-Aware-MT

---
## 2 — Pre-flight

Every assertion here guards a way this run could silently corrupt the primary Φ
or produce an invalid comparison. Do not relax one to make the cell pass.

In [ ]:
import json, pathlib, yaml
from src.eval.judge import judge_results_path, judge_segment_dir, template_digest
from src.eval.judge_agreement import REFERENCE_CONDITION, STUDY_CONDITIONS

CONFIG  = 'configs/judge_eval_gpt.yaml'
SPLIT   = 'val'
CONDS   = [*STUDY_CONDITIONS, REFERENCE_CONDITION]      # the seven main conditions
RESULTS = pathlib.Path('results')

cfg = yaml.safe_load(pathlib.Path(CONFIG).read_text())
judge, TAG = cfg['judge'], cfg['tag']

# -- the second rater must actually be a different family, or there is no cross-check
assert judge['provider'] == 'openai', judge['provider']
assert TAG, 'tag must be set, or this run overwrites the primary judge artefacts'

# -- identical frozen rubric: rater identity must be the only thing that differs
assert cfg['template_file'] == 'prompts/judge_eval.txt', cfg['template_file']
DIGEST = template_digest(pathlib.Path(cfg['template_file']).read_text())

# -- artefacts must not collide with the primary judge's
out_b, dir_b = judge_results_path(RESULTS, SPLIT, TAG), judge_segment_dir(RESULTS, SPLIT, TAG)
out_a, dir_a = judge_results_path(RESULTS, SPLIT, None), judge_segment_dir(RESULTS, SPLIT, None)
assert out_b != out_a and dir_b != dir_a
assert out_a.exists(), 'the primary judge results are missing; nothing to compare against'

# -- all seven conditions present, equal length, identical segments in identical order
rows = {c: [json.loads(x) for x in pathlib.Path(f'outputs/{c}_{SPLIT}.jsonl')
            .read_text().splitlines() if x.strip()] for c in CONDS}
n_eval = len(rows[CONDS[0]])
for c in CONDS:
    assert len(rows[c]) == n_eval, f'{c}: {len(rows[c])} rows, expected {n_eval}'
    assert [r['input'] for r in rows[c]] == [r['input'] for r in rows[CONDS[0]]], \
        f'{c} segments diverge from {CONDS[0]}; the paired comparison requires alignment'

print(f'conditions   : {len(CONDS)}  {CONDS}')
print(f'segments each: {n_eval}   (test split sealed)')
print(f'judge B      : {judge["model"]}  tag={TAG}')
print(f'rubric       : {cfg["template_file"]} [{DIGEST}]  (same file as judge A)')
print(f'writes to    : {out_b}  and  {dir_b}/')
print(f'judge A safe : {out_a} untouched')

In [ ]:
# The primary judge's own results file must be intact and from the OTHER model.
# assert_results_identity refuses the overwrite at run time; this shows it now.
a = json.loads(out_a.read_text())
models_a = sorted({v['model'] for v in a.values() if isinstance(v, dict)})
print('judge A model(s):', models_a)
print('judge A conditions:', sorted(a))
assert judge['model'] not in models_a, 'judge B is the same model as judge A'
missing = set(CONDS) - set(a)
print('conditions judge A has not scored:', missing or 'none')

---
## 3 — Cost: measure it, do not guess it

Judge calls are the only paid component of this project. The full pass is
**7 × 1,323 = 9,261 calls**, and with a reasoning model the *hidden* reasoning
tokens dominate the output bill and cannot be estimated reliably from the prompt.

So: static estimate → **25-call pilot** → extrapolate from real usage → decide.
Do not skip to section 4.

In [ ]:
# Static estimate from the real prompts. Input side only -- it is the half that
# can actually be counted in advance.
tpl = pathlib.Path(cfg['template_file']).read_text()
chars = sum(len(tpl) + len(r['input']) + len(r['output']) + len(r.get('prediction', ''))
            for c in CONDS for r in rows[c])
calls = len(CONDS) * n_eval
in_rate, out_rate = cfg['judge'].get('pricing', (None, None))

print(f'calls            : {calls:,}')
print(f'mean prompt chars: {chars / calls:,.0f}')
for div in (3.0, 3.5, 4.0):
    tok = chars / div
    cost = tok / 1e6 * in_rate if in_rate else float("nan")
    print(f'  @{div} chars/tok -> {tok/1e6:5.2f}M input tokens  ~${cost:6.2f} input-side')
print('\nOutput-side cost is UNKNOWN until the pilot: reasoning tokens are billed',
      'as output and are not visible in the prompt.')

In [ ]:
# PILOT -- 25 segments of one condition. --limit writes the segment cache but
# deliberately does NOT write a results file, so this cannot be mistaken for a
# scored condition. The full run below resumes from these 25 at no extra cost.
!python3 manage.py judge --conditions zeroshot --split {SPLIT} --config {CONFIG} --limit 25

In [ ]:
# Extrapolate from what the pilot actually spent.
u = json.loads((RESULTS / f'judge_{TAG}_{SPLIT}_usage.json').read_text())
s = u['session']
per_call = {k: s[k] / s['calls'] for k in ('prompt_tokens', 'completion_tokens', 'cost_usd')}

print(f"pilot: {s['calls']} calls, {s['prompt_tokens']:,} in / {s['completion_tokens']:,} out")
print(f"       ${s['cost_usd']:.4f}  ->  ${per_call['cost_usd']:.5f} per call")
print(f"       output/input token ratio {s['completion_tokens'] / max(s['prompt_tokens'], 1):.2f}")
if not u['priced']:
    print('\n!! no pricing configured for this model -- cost_usd is a floor of 0, not real spend')

projected = per_call['cost_usd'] * calls
print(f'\nPROJECTED FULL PASS: {calls:,} calls  ~${projected:.2f}')
print('\nStop here. Confirm this against the declared budget cap before running',
      'section 4. If the projection is unacceptable, the documented fallback is to',
      'score a stratified subset -- but a subset changes n and every interval with it.')

---
## 4 — Score the seven conditions

Resumable: the segment cache is flushed and fsynced per call, so an interrupted
run continues where it stopped. Re-running after completion costs nothing.

The cache is bound to `(model, tag, rubric digest)` — pointing this at the primary
judge's cache raises rather than silently returning that judge's scores.

In [ ]:
CONDS_ARG = ' '.join(CONDS)
!python3 manage.py judge --conditions {CONDS_ARG} --split {SPLIT} --config {CONFIG}

In [ ]:
# Coverage audit: an unparsed score is dropped, not imputed, so coverage below
# 100% shifts n for that condition and must be carried through explicitly.
b = json.loads(out_b.read_text())
print(f"{'condition':<18}{'n':>6}{'coverage':>10}{'Phi_B':>8}{'Phi_A':>8}{'B-A':>8}")
for c in CONDS:
    rb, ra = b.get(c), a.get(c)
    if not rb:
        print(f'{c:<18}  MISSING'); continue
    d = rb['mean'] - ra['mean'] if ra and ra['mean'] and rb['mean'] else float('nan')
    print(f"{c:<18}{rb['n']:>6}{rb['coverage']:>10.4f}{rb['mean']:>8.3f}"
          f"{ra['mean'] if ra else float('nan'):>8.3f}{d:>+8.3f}")

errs = {c: sum('error' in json.loads(x) for x in (dir_b / f'{c}.jsonl').read_text().splitlines() if x.strip())
        for c in CONDS if (dir_b / f'{c}.jsonl').exists()}
print('\nfailed calls per condition:', {k: v for k, v in errs.items() if v} or 'none')
print('re-run the cell above to retry failures; completed segments are not re-billed')

---
## 5 — Judge–judge agreement

Paired at segment level on identical segments. Contrasts are held to one common
segment set across *both* raters, so a difference between the two columns is rater
identity and nothing else.

In [ ]:
!python3 manage.py judge_agreement --tag_b {TAG} --split {SPLIT} --n_resamples 10000

In [ ]:
# Headline read-out: which pre-specified conclusions survive the rater swap.
rep = json.loads((RESULTS / f'judge_agreement_{TAG}_{SPLIT}.json').read_text())

pooled = rep['rater_agreement']['study_only']['pooled']
print(f"pooled n={pooled['n']}  qwk={pooled['qwk']['kappa']:+.3f}"
      f"  rho={pooled['spearman']['rho']:+.3f}"
      f"  exact={pooled['exact_agreement']:.1%}  adjacent={pooled['adjacent_agreement']:.1%}")
print(f"severity offset A-B = {pooled['offset']['diff']:+.3f}"
      f" [{pooled['offset']['ci_low']:+.3f}, {pooled['offset']['ci_high']:+.3f}]")

co = rep['condition_ordering']['study_only']
print(f"\nidentical system ranking: {co['identical_ranking']}")

print('\nContrasts by stability:')
for verdict in ('RATER-DEPENDENT', 'both separate', 'neither separates'):
    names = [k for k, v in rep['contrast_replication']['contrasts'].items()
             if (('RATER-DEPENDENT' if not (v['both_separate'] or v['neither_separates'])
                  else 'both separate' if v['both_separate'] else 'neither separates') == verdict)]
    print(f'  {verdict:<18} {names or "-"}')

flipped = [k for k, v in rep['contrast_replication']['contrasts'].items() if not v['same_sign']]
print(f'\nsign flips between raters: {flipped or "none"}')

---
## 6 — What may be reported from this

Read the output above against these rules before writing any of it up.

- A contrast marked **`both separate`** (Holm-corrected under each rater) is
  robust to rater identity. Report it with both intervals.
- A contrast marked **`RATER-DEPENDENT`** separates under one rater only. It is
  not reportable as a finding without naming the rater it depends on.
- **`neither separates`** is a failure to separate. It is *not* evidence of no
  difference, and the detection floor still applies.
- A **sign flip** between raters means the direction itself is not established.
- Agreement is **descriptive**. A high κ does not validate Φ; it shows two raters
  behave similarly, which is also what two similarly-biased raters would do.
- Φ_B is **not** byte-reproducible either. The seed is best-effort, so Φ_B
  differences of the same order as Φ_A's noise are within measurement noise.
- The severity offset shifts every condition together, so it cannot create or
  destroy a contrast — but it does mean the two Φ columns are not interchangeable
  in any table reporting absolute values. Do not average them into one Φ.
- `commercial_haiku` remains a **diagnostic external reference**, not a condition
  of the study, under either rater. If judge A scores it markedly higher relative
  to the study conditions than judge B does, that gap is the self-preference
  effect and belongs in the threats-to-validity register.

Update on completion: `docs/DEVLOG.md` (what ran, actual spend, what it showed),
the threats table in `README.md` §validity (single-judge dependence is no longer
unaddressed), and `docs/budget.md` with the cumulative figure from the usage file.